4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date
and is_current) and inserts new versions when a tracked column changes.

In [0]:
%python

from pyspark.sql.functions import *

source_path = "/Volumes/dev/bronze/raw/products/"
checkpoint_path = "/Volumes/dev/bronze/raw/checkpoints/"
schema_path = "/Volumes/dev/bronze/raw/schama/"

bronze_table = "dev.bronze.products"
silver_table = "dev.silver.products"

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)

    # Metadata
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn(
        "source_file_name",
        element_at(split(col("_metadata.file_path"), "/"), -1)
    )
    .withColumn("ingestion_timestamp", current_timestamp())

    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_table = "dev.bronze.products"
silver_table = "dev.silver.products"

# Read Bronze
bronze_df = spark.table(bronze_table)
bronze_df = bronze_df.drop("_rescued_data", "source_file_path", "source_file_name")

# Find only the new records which are not processed into Silver yet
if spark.catalog.tableExists(silver_table):
    last_ingestion = (
        spark.table(silver_table)
        .agg(F.max("bronze_ingestion_timestamp").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
else:
    last_ingestion = None

if last_ingestion is None:
    source_df = bronze_df
else:
    source_df = bronze_df.filter(
        F.col("ingestion_timestamp") > F.lit(last_ingestion)
    )

# Clean
clean_df = (
    source_df
    .withColumn("product_id", F.col("product_id"))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("price", F.col("price").cast("double"))
)

# Validation
clean_df = clean_df.filter(
    F.col("product_id").isNotNull() &
    F.col("product_name").isNotNull() &
    F.col("category").isNotNull() &
    F.col("price").isNotNull() &
    (F.col("price") >= 0)
)

In [0]:
%python
if not spark.catalog.tableExists(silver_table):

    silver_df = (
        clean_df
        .select(
            "product_id",
            "product_name",
            "category",
            "price",
            F.col("ingestion_timestamp").cast("string").alias("bronze_ingestion_timestamp")
        )
        .withColumn("effective_start_date", F.current_date())
        .withColumn(
            "effective_end_date",
            F.lit(None).cast("date")
        )
        .withColumn("is_current", F.lit(True))
    )

    (
        silver_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

clean_df.createOrReplaceTempView("daily_products")

In [0]:
MERGE INTO dev.silver.products AS target

USING daily_products AS source

ON target.product_id = source.product_id
AND target.is_current = true

WHEN MATCHED AND (
       target.product_name <> source.product_name
    OR target.category <> source.category
    OR target.price <> source.price
)

THEN UPDATE SET
    target.effective_end_date = current_date(),
    target.is_current = false;



In [0]:
INSERT INTO dev.silver.products
(
    product_id,
    product_name,
    category,
    price,
    bronze_ingestion_timestamp,
    effective_start_date,
    effective_end_date,
    is_current
)

SELECT
    s.product_id,
    s.product_name,
    s.category,
    s.price,
    s.ingestion_timestamp,
    current_date(),
    NULL,
    true

FROM daily_products s

LEFT JOIN dev.silver.products t
    ON s.product_id = t.product_id
   AND t.is_current = true

WHERE t.product_id IS NULL;

5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's
address as of March 1st?'

-->
I have performed **SCD Type 2** on the `products` table.

Using SCD Type 2, I can check the **price or any other attribute of a product at a particular date** and also see the complete history of that product.

For example, for **product_id = "P0008" **, I can check its price as of a specific date.

### Check product 5 history

```sql
SELECT *
FROM dev.silver.products
WHERE product_id = "P0008"
ORDER BY effective_start_date;

In [0]:
SELECT *
FROM dev.silver.products
WHERE product_id = "P0008"
ORDER BY effective_start_date;

6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend
which Cyntexa should use for its nightly pipeline.

-->
%md

### DBU Cost Comparison

The actual DBU price can vary depending on the cloud platform, region, Databricks pricing tier, and the pricing agreement with Cyntexa.

For example, if we consider an AWS Premium setup, All-Purpose Compute is roughly **$0.55 per DBU-hour**, while Job Compute is around **$0.30 per DBU-hour**.

This means Job Compute is cheaper for the same amount of DBU usage. For a nightly pipeline that runs automatically and does not need an interactive cluster, using Job Compute can help reduce the overall compute cost.

### Recommendation

For Cyntexa's nightly pipeline, I would recommend using **Job Compute**.

The pipeline is a scheduled production workload, so there is no real need to keep an interactive All-Purpose cluster running. Job Compute is designed for this type of workload and automatically terminates when the job is finished.

All-Purpose Compute would be more suitable for **development, testing, debugging, and interactive notebook work**.

Therefore, **Job Compute is the better choice for the nightly pipeline because it is more cost-efficient and better suited for scheduled production jobs.**